# 02: Preprocessing (Fixed v2)
Parse 6 ATT&CK technique log files.

In [ ]:
import pandas as pd
import numpy as np
import json, os, glob, re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
print('=== NOTEBOOK 02: PREPROCESSING ===')
RAW_DIR = '../data/raw'
PROCESSED_DIR = '../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)
TECHNIQUE_MAP = {
    'T1566_Phishing': 'T1566',
    'T1110_BruteForce': 'T1110',
    'T1059_CommandShell': 'T1059',
    'T1055_ProcessInjection': 'T1055',
    'T1082_SystemInfo': 'T1082',
    'T1021_RemoteServices': 'T1021',
}


In [ ]:
def parse_log_line(line):
    line = line.strip()
    if not line:
        return None
    try:
        result = json.loads(line)
        if isinstance(result, dict):
            return result
    except:
        pass
    event = {}
    matches = re.compile(r'(\w+)[=:]([^\s]+|"[^"]*")').findall(line)
    if matches:
        for key, val in matches:
            event[key] = val.strip('"')
        return event if len(event) > 2 else None
    return None

all_records = []
for folder_name, technique_id in TECHNIQUE_MAP.items():
    folder_path = os.path.join(RAW_DIR, folder_name)
    if not os.path.exists(folder_path):
        print(f'WARNING: {folder_path} not found')
        continue
    log_files = glob.glob(os.path.join(folder_path, '*.log'))
    print(f'Processing {technique_id}: {len(log_files)} file(s)')
    for log_file in log_files:
        count = 0
        with open(log_file, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                event = parse_log_line(line)
                if event is not None:
                    event['technique_id'] = technique_id
                    all_records.append(event)
                    count += 1
        print(f'  {os.path.basename(log_file)}: {count} events')

print(f'\nTotal raw events: {len(all_records)}')
if len(all_records) == 0:
    raise ValueError('No events parsed!')


In [ ]:
df = pd.DataFrame(all_records)
print(f'DataFrame shape: {df.shape}')
print(f'Columns: {list(df.columns)}')


In [ ]:
print('\nExtracting features...')
features = pd.DataFrame()
n = len(df)

# Helper to get column or empty string series
def get_col(df, col):
    return df[col].astype(str) if col in df.columns else pd.Series([''] * n, index=df.index)

features['raw_length'] = df.apply(lambda row: len(str(row)), axis=1)
features['EventCode'] = pd.to_numeric(get_col(df, 'EventCode'), errors='coerce').fillna(0)

source = get_col(df, 'sourcetype') if 'sourcetype' in df.columns else get_col(df, 'SourceName')
features['source_type'] = source

if 'Image' in df.columns:
    process = df['Image'].astype(str)
elif 'ProcessName' in df.columns:
    process = df['ProcessName'].astype(str)
elif 'NewProcessName' in df.columns:
    process = df['NewProcessName'].astype(str)
else:
    process = pd.Series([''] * n, index=df.index)
features['process_name'] = process
features['process_name_length'] = features['process_name'].str.len()
features['has_process'] = (features['process_name'] != '').astype(int)

cmd = get_col(df, 'CommandLine')
features['command_line'] = cmd
features['command_line_length'] = features['command_line'].str.len()
features['has_command_line'] = (features['command_line'] != '').astype(int)

if 'ParentImage' in df.columns:
    parent = df['ParentImage'].astype(str)
elif 'ParentProcessName' in df.columns:
    parent = df['ParentProcessName'].astype(str)
else:
    parent = pd.Series([''] * n, index=df.index)
features['has_parent_process'] = (parent != '').astype(int)

if 'User' in df.columns:
    user = df['User'].astype(str)
elif 'AccountName' in df.columns:
    user = df['AccountName'].astype(str)
elif 'SubjectUserName' in df.columns:
    user = df['SubjectUserName'].astype(str)
else:
    user = pd.Series([''] * n, index=df.index)
features['user_present'] = (user != '').astype(int)

if 'Computer' in df.columns:
    computer = df['Computer'].astype(str)
elif 'ComputerName' in df.columns:
    computer = df['ComputerName'].astype(str)
else:
    computer = pd.Series(['unknown'] * n, index=df.index)
features['computer'] = computer

if '_time' in df.columns:
    time_parsed = pd.to_datetime(df['_time'], errors='coerce', utc=True)
elif 'TimeCreated' in df.columns:
    time_parsed = pd.to_datetime(df['TimeCreated'], errors='coerce', utc=True)
else:
    time_parsed = pd.Series([pd.NaT] * n, index=df.index)
features['hour_of_day'] = time_parsed.dt.hour.fillna(0)
features['day_of_week'] = time_parsed.dt.dayofweek.fillna(0)

if 'Channel' in df.columns:
    log_type = df['Channel'].astype(str)
elif 'LogName' in df.columns:
    log_type = df['LogName'].astype(str)
else:
    log_type = pd.Series(['unknown'] * n, index=df.index)
features['log_type'] = log_type

if 'TargetObject' in df.columns:
    target = df['TargetObject'].astype(str)
elif 'TargetFilename' in df.columns:
    target = df['TargetFilename'].astype(str)
else:
    target = pd.Series([''] * n, index=df.index)
features['has_target'] = (target != '').astype(int)

features['num_keys'] = df.apply(lambda row: len([k for k in row.keys() if pd.notna(row[k]) and str(row[k]) != '']), axis=1)
features['technique_id'] = df['technique_id'].values

print(f'Feature matrix shape: {features.shape}')
print(f'Features: {list(features.columns)}')


In [ ]:
le_technique = LabelEncoder()
features['technique_encoded'] = le_technique.fit_transform(features['technique_id'])

for col in ['source_type', 'process_name', 'command_line', 'computer', 'log_type']:
    le = LabelEncoder()
    features[col] = le.fit_transform(features[col].astype(str))

X = features.drop(['technique_id', 'technique_encoded'], axis=1)
y = features['technique_encoded']

print(f'\nClass distribution:')
print(y.value_counts().sort_index())
print(f'\nTotal classes: {y.nunique()}')


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train classes: {sorted(y_train.unique())}')
print(f'Test classes: {sorted(y_test.unique())}')

X_train.to_csv(f'{PROCESSED_DIR}/X_train.csv', index=False)
X_test.to_csv(f'{PROCESSED_DIR}/X_test.csv', index=False)
y_train.to_csv(f'{PROCESSED_DIR}/y_train.csv', index=False)
y_test.to_csv(f'{PROCESSED_DIR}/y_test.csv', index=False)

label_map_df = pd.DataFrame({'encoded': range(len(le_technique.classes_)), 'technique': le_technique.classes_})
label_map_df.to_csv(f'{PROCESSED_DIR}/label_map.csv', index=False)
pd.DataFrame({'feature': X.columns}).to_csv(f'{PROCESSED_DIR}/feature_names.csv', index=False)

print(f'\nSaved to {PROCESSED_DIR}/')
print(label_map_df.to_string(index=False))
print('\n=== PREPROCESSING COMPLETE ===')
